In [ ]:
import os, sys, glob
import numpy as np
import sunpy.map
import drms
from pathlib import Path

from matplotlib import pyplot as plt
import matplotlib.colors as mcolor
from matplotlib.backends.backend_pdf import PdfPages


In [ ]:
from astropy.io import fits
from astropy.convolution import convolve
from astropy import units as u
from astropy import constants as const
from astropy.coordinates import SkyCoord

from scipy.special import j1

import pfsspy
from pfsspy import tracing

In [ ]:
SRC_PATH = (Path.cwd() / "../src").resolve()
sys.path.insert(0, str(SRC_PATH))

In [ ]:
#%matplotlib widget
%matplotlib inline

# Code

##  Path definitions

In [ ]:
carrington_number = 2287 # 2284, 2285, 2286, 2287

In [ ]:
root = Path.cwd()
path = root / '../data/pfss_test/'
fname_phi = f'synopMr_CR{carrington_number}.fits'



In [ ]:
fname = "synopMr.fits"
series="hmi.synoptic_mr_polfil_720s"
segment="Mr_polfil"
small = False

In [ ]:
############################
####### GET PHI DATA #######
############################

synop_phi  = fits.open(path / fname_phi)

phi_img   = synop_phi[0].data
if not small: 
    phi_table = synop_phi[1].data

# Define Carrington rotation number
#carrington_number = int(synop_phi[0].header["CAR_ROT"])
outname = f"{carrington_number}"


############################
####### GET HMI DATA #######
############################

# Create DRMS client
c = drms.Client(email="loeschl@mps.mpg.de")#, verbose=True)

datapath_hmi = os.path.join(root, f"../data/tmp/CR{carrington_number}/")
os.makedirs(datapath_hmi, exist_ok=True)

# Query JSOC for that rotation
q = c.query(f"{series}[{carrington_number}]", seg=segment)
fname_hmi = q[segment][0].split('/')[-1]

try:
    file_hmi  = glob.glob(f"{datapath_hmi}*{fname_hmi}")[0]
    synop_hmi = fits.open(file_hmi)
except IndexError:
    # Download the FITS file
    result = c.export(f"{series}[{carrington_number}]", method='url', protocol='fits')
    result.download(datapath_hmi)
    file_hmi  = glob.glob(f"{datapath_hmi}*{fname_hmi}")[0]
    synop_hmi = fits.open(file_hmi)

try: 
    hmi_img = synop_hmi[1].data   

    # create mask of NaN values in phi_img and fill them with hmi_img values
    mask_polfil = np.isnan(phi_img)
    phi_polfil  = np.where(mask_polfil, hmi_img, phi_img)    

    # aggressive HMI pole filling
    #phi_polfil[:40, :]  = hmi_img[:40, :]
    #phi_polfil[1400: :] = hmi_img[1400:, :]

    #synop_phi[0].header["CUNIT2"] = "deg" # "Sine Latitude"

    primary_hdu = fits.PrimaryHDU()

    polfil_hdu = fits.CompImageHDU(data=phi_polfil, 
                                    header=synop_phi[0].header, 
                                    compression_type='RICE_1')

    polfil_hdul = fits.HDUList([primary_hdu, polfil_hdu])
    polfil_hdul.writeto(os.path.join(path, "synopMr_polfil.fits"), overwrite=True)

    # show filled synoptic map 
    #plt.imshow(phi_polfil, cmap='hmimag', vmin=-1500, vmax=1500, origin='lower')
    #plt.show()

except IndexError:
    # non polfil data
    hmi_img = synop_hmi[0].data

## Function definitions

In [ ]:
def airy_psf(size, radius_px):
    y, x = np.indices((size, size))
    r = np.sqrt((x - size//2)**2 + (y - size//2)**2)
    kr = np.pi * r / radius_px
    psf = np.ones_like(r)
    mask = kr != 0
    psf[mask] = (2 * j1(kr[mask]) / kr[mask])**2
    return psf / psf.sum()

In [ ]:
def airy_psf_v2(size, fwhm):
    """
    Create an Airy disk PSF.
    
    size: int, width/height of PSF kernel (pixels)
    fwhm: float, FWHM in pixels
    """
    y, x = np.indices((size, size)) - size // 2
    r = np.sqrt(x**2 + y**2)
    
    # Convert FWHM to first zero (approximation)
    r0 = fwhm / 1.028  # Airy FWHM ≈ 1.028 * lambda / D
    
    r_norm = np.pi * r / r0
    psf = (2 * j1(r_norm) / r_norm)**2
    psf[r==0] = 1.0  # handle division by zero at center
    psf /= psf.sum()
    return psf

In [ ]:
def pfss(phi_polfil, synop_hmi):
    ###############################################################################
    # Since this map is far to big to calculate a PFSS solution quickly, lets
    # resample it down to a smaller size.

    #phi_map.meta["CUNIT2"] = "deg" # "Sine Latitude"

    # pfsspy sometimes struggles with PHI header, replace with compatible hmi header
    phi_map = sunpy.map.Map(phi_polfil, dict(synop_hmi[1].header))
    #phi_map = phi_map.resample([720, 360] * u.pix)
    phi_map = phi_map.resample([480, 240] * u.pix)
    #phi_map = phi_map.resample([360, 180] * u.pix)

    #print('New shape: ', phi_map.data.shape)

    ###############################################################################
    # Now calculate the PFSS solution
    #nrho = 25
    nrho = 50
    rss = 2.5
    pfss_in = pfsspy.Input(phi_map, nrho, rss)
    pfss_out = pfsspy.pfss(pfss_in)

    ###############################################################################
    # Finally, using the 3D magnetic field solution we can trace some field lines.
    # In this case a grid of 90 x 180 points equally gridded in theta and phi are
    # chosen and traced from the source surface outwards.
    #
    # First, set up the tracing seeds

    r = const.R_sun
    # Number of steps in cos(latitude)
    nsteps = 90
    lon_1d = np.linspace(0, 2 * np.pi, nsteps * 2 + 1)
    lat_1d = np.arcsin(np.linspace(-1, 1, nsteps + 1))
    lon, lat = np.meshgrid(lon_1d, lat_1d, indexing='ij')
    lon, lat = lon*u.rad, lat*u.rad
    seeds = SkyCoord(lon.ravel(), lat.ravel(), r, frame=pfss_out.coordinate_frame)

    ###############################################################################
    # Trace the field lines
    print('Tracing field lines...')
    tracer = tracing.FortranTracer(max_steps=5000)
    field_lines = tracer.trace(seeds, pfss_out)
    print('Finished tracing field lines')

    ###############################################################################
    # Plot the result. The to plot is the input magnetogram, and the bottom plot
    # shows a contour map of the the footpoint polarities, which are +/- 1 for open
    # field regions and 0 for closed field regions.

    fig = plt.figure(figsize=(8,11.25))

    # --- First subplot ---
    ss_br = pfss_out.source_surface_br
    ax1 = fig.add_subplot(3, 1, 1, projection=ss_br)
    im1 = ss_br.plot()
    ax1.plot_coord(pfss_out.source_surface_pils[0])
    ax1.set_title(f'Source surface magnetic field')
    plt.colorbar(im1, ax=ax1)  # Attach colorbar to ax1

    # --- Second subplot ---
    ax2 = fig.add_subplot(3, 1, 2)
    cmap = mcolor.ListedColormap(['tab:red', 'black', 'tab:blue'])
    norm = mcolor.BoundaryNorm([-1.5, -0.5, 0.5, 1.5], ncolors=3)
    pols = field_lines.polarities.reshape(2 * nsteps + 1, nsteps + 1).T
    cf2 = ax2.contourf(np.rad2deg(lon_1d), np.sin(lat_1d), pols, norm=norm, cmap=cmap)
    ax2.set_ylabel('sin(latitude)')
    ax2.set_title('Open (blue/red) and closed (black) field')
    ax2.set_aspect(0.5 * 360 / 2)
    plt.colorbar(cf2, ax=ax2)  # Attach colorbar to ax2

    # --- Third subplot ---
    m = pfss_in.map # Create a norm with the limits you want 
    norm = m.plot_settings['norm'] # get existing norm 
    norm.vmin = None #-1500 # reset vmin 
    norm.vmax = None #+1500 # reset vmax
    ax3 = fig.add_subplot(3, 1, 3, projection=m)
    im3 = m.plot(cmap='hmimag', vmin=-1500, vmax=1500)
    xx = pfss_in.map.data.shape[1]/360
    yy = pfss_in.map.data.shape[0]//2
    ax3.contourf(np.rad2deg(lon_1d)*xx, np.sin(lat_1d)*yy+yy, pols, norm=norm, cmap=cmap, alpha=0.25)
    ax3.plot_coord(pfss_out.source_surface_pils[0])
    ax3.set_title(f'PHI/HMI input magnetogram w/ PFSS & Open Field')
    plt.colorbar(im3, ax=ax3)  # Attach colorbar to ax3




## Analysis

In [ ]:
hmi_segment = synop_phi[1].data[synop_phi[1].data['SRC'] == 'HMI']
hmi_segment

In [ ]:
# table is going right to left  -> invert
hmi_end = int(hmi_segment['CRLN_START'][0]*10)
hmi_start = int(hmi_segment['CRLN_END'][0]*10)

if hmi_start > hmi_end:
    hmi_end = 3600

hmi_start, hmi_end

In [ ]:
plt.close()
fig, ax = plt.subplots(figsize=(10, 8), nrows=1, ncols=1, sharex=True, sharey=True)
im1 = ax.imshow(phi_polfil[:,:hmi_start], vmin=-1500, vmax=1500, cmap="hmimag", interpolation='None', origin='lower', label='PHI CR')

ax.set_title('PHI/HMI combined synoptic map - PHI resolution')

ax.set_facecolor('lightgray')

fig.colorbar(im1, label='Br [G]')

In [ ]:
plt.close()
fig, ax = plt.subplots(figsize=(10, 8), nrows=1, ncols=1, sharex=True, sharey=True)
im1 = ax.imshow(phi_polfil[:,hmi_start:hmi_end], vmin=-1500, vmax=1500, cmap="hmimag", interpolation='None', origin='lower', label='PHI CR')

ax.set_title('PHI/HMI combined synoptic map - HMI resolution')

ax.set_facecolor('lightgray')

fig.colorbar(im1, label='Br [G]')

## PSF Kernel

In [ ]:
# Set up pixelscale info from cdelt keyword (currently manual!)
cdelt_phi_cr2284 = 3.5748234469 # CR 2284
cdelt_phi_cr2285 = 3.5748234469 # CR 2285
cdelt_phi_cr2286 = 3.5748234469 # CR 2286
cdelt_phi_cr2287 = 3.5748234469 # CR 2287

cdelt_hmi = 0.504032
cdelt_phi = cdelt_phi_cr2284

In [ ]:
theta_fdt = (1.22 * (617 * 1e-9)/0.0175)#*(24*3600)
theta_fdt_arcsec = np.degrees(theta_fdt)*(3600) 
theta_fdt_arcsec

In [ ]:
theta_hmi = (1.22 * (617 * 1e-9)/0.14)#*(24*3600)
theta_hmi_arcsec = np.degrees(theta_hmi)*(3600) 
theta_hmi_arcsec

In [ ]:
# 1800 px in remapped xdim. 3820 px solar disk in HMI: 1910 arcsec solar diameter / 0.504 arcsec/px = 3820 px
fwhm_existing = 3820/1800
print(f'HMI plate scale on reprojected/remapped frame {fwhm_existing:.4f} arc/px') 

In [ ]:
fwhm_final = theta_fdt_arcsec/theta_hmi_arcsec
print(f'FWHM difference between HMI and PHI airy disks is a factor {fwhm_final}')

In [ ]:
# FWHM_final = FWHM_existing + FWHM_kernel
fwhm_kernel = np.sqrt(fwhm_final**2 - fwhm_existing**2)

print(f"Remaining blur as FWHM kernel size {fwhm_kernel:.2f} px")

In [ ]:
size = 65
halfsize = size//2
size, halfsize

In [ ]:
psf_to_fdt  = airy_psf_v2(size, fwhm_kernel)

In [ ]:
plt.imshow(psf_to_fdt, origin='lower', cmap='viridis')

In [ ]:
if hmi_start < halfsize:
    hmi_start = halfsize
    
if hmi_end == 3600:
    hmi_end = 3600 - halfsize
    
hmi_conv = convolve(phi_polfil[:,hmi_start-halfsize:hmi_end+halfsize], psf_to_fdt, boundary='fill', fill_value=np.nan, normalize_kernel=True)

In [ ]:
plt.close()
fig, ax = plt.subplots(figsize=(10, 8), nrows=2, ncols=1, sharex=True, sharey=True)
im1 = ax[0].imshow(hmi_conv, vmin=-1500, vmax=1500, cmap="hmimag", interpolation='None', origin='lower', label='PHI CR')
im2 = ax[1].imshow(phi_polfil[:,hmi_start-halfsize:hmi_end+halfsize], vmin=-1500, vmax=1500, cmap="hmimag", interpolation='None', origin='lower', label='PHI CR')

ax[0].set_title('PHI/HMI combined synoptic map - PHI resolution')
ax[1].set_title('PHI/HMI combined synoptic map - HMI resolution')

ax[0].set_facecolor('lightgray')
ax[1].set_facecolor('lightgray')

fig.colorbar(im1, label='Br [G]')
fig.colorbar(im2, label='Br [G]')


In [ ]:
phi_polfil_degr = phi_polfil.copy()

In [ ]:
phi_polfil_degr[halfsize:-halfsize,hmi_start:hmi_end] = hmi_conv[halfsize:-halfsize,halfsize:-halfsize]

In [ ]:
plt.close()
fig, ax = plt.subplots(figsize=(10, 5), nrows=1, ncols=1, sharex=True, sharey=True)
im1 = ax.imshow(phi_polfil_degr, vmin=-1500, vmax=1500, cmap="hmimag", interpolation='None', origin='lower', label='PHI CR')

ax.set_title('PHI/HMI combined synoptic map - HMI resolution')

ax.set_facecolor('lightgray')

fig.colorbar(im1, label='Br [G]')

In [ ]:
# why is there no flux getting deleted here?!

phi_degr_seg = phi_polfil_degr[halfsize:-halfsize,hmi_start:hmi_end].flatten()
phi_seg      = phi_polfil[halfsize:-halfsize,hmi_start:hmi_end].flatten()

phi_pos = np.nansum(phi_seg[phi_seg > 0].flatten())
phi_neg = np.nansum(phi_seg[phi_seg < 0].flatten())

phi_degr_pos = np.nansum(phi_degr_seg[phi_degr_seg > 0].flatten())
phi_degr_neg = np.nansum(phi_degr_seg[phi_degr_seg < 0].flatten())

phi_unsigned = phi_pos + np.abs(phi_neg)
phi_degr_unsigned = phi_degr_pos + np.abs(phi_degr_neg)

In [ ]:
phi_degr_unsigned/phi_unsigned, phi_unsigned/phi_degr_unsigned

## PFSS Tests

In [ ]:
pfss(phi_polfil, synop_hmi)

In [ ]:
pfss(phi_polfil_degr, synop_hmi)

In [ ]:
pfss(hmi_img, synop_hmi)